# 02 — Statistical Hypothesis Testing

This notebook formalizes what section 3 of `01_eda.ipynb` explored: for every
feature, is its association with diabetes status statistically real, how
large is it, and does it survive correction for running many tests at once?

Structure:
1. Hypotheses and test plan
2. Assumption checks (normality, variance homogeneity)
3. Hypothesis tests — categorical features (chi-square)
4. Hypothesis tests — continuous features (Mann-Whitney U)
5. Multiple-testing correction
6. Consolidated results table
7. Interpretation

In [9]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.data_loader import load_raw_data
from src.utils import load_config, set_seed
from src.eda import chi_square_test, mann_whitney_test, cramers_v
from src.hypothesis_testing import (
    check_normality, check_variance_homogeneity,
    rank_biserial_from_mannwhitney, cohens_d,
    apply_multiple_testing_correction, effect_size_label,
)

config = load_config()
set_seed(config["random_seed"])

df = load_raw_data()
target = config["data"]["target_column"]

binary_cols = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "HeartDiseaseorAttack",
    "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare",
    "NoDocbcCost", "DiffWalk", "Sex",
]
ordinal_cols = ["GenHlth", "Age", "Education", "Income"]
continuous_cols = ["BMI", "MentHlth", "PhysHlth"]
categorical_cols = binary_cols + ordinal_cols

2026-09-18 16:08:36 | src.data_loader | INFO | Loading cached raw data from C:\diabetes-risk-prediction\data\raw\diabetes_health_indicators.csv
2026-09-18 16:08:37 | src.data_loader | INFO | Raw data validation passed.


## 1. Hypotheses and Test Plan

State upfront what's being tested and how, before running anything — this is
what makes it a confirmatory pass rather than more exploration.

**For every categorical/binary/ordinal feature X:**
- H0: X is independent of diabetes status (no association)
- H1: X is associated with diabetes status
- Test: Chi-square test of independence
- Effect size: Cramer's V

**For every continuous feature X (BMI, MentHlth, PhysHlth):**
- H0: X has the same distribution in both diabetes groups
- H1: X's distribution differs between groups
- Test: Mann-Whitney U (non-parametric — see assumption checks in section 2
  for why a t-test isn't used here)
- Effect size: rank-biserial correlation (primary), Cohen's d (reported for
  context only, since it assumes normality these features don't have)

**Significance threshold:** alpha = 0.05, corrected for multiple comparisons
via Benjamini-Hochberg FDR across all 21 tests run in this notebook — 14
binary + 4 ordinal + 3 continuous features (see
section 5) — a feature only counts as a confirmed finding if it survives
that correction, not just the raw per-test p-value.

## 2. Assumption Checks

Checking whether the continuous features meet the assumptions a t-test would
need (normality, equal variance) — this justifies using Mann-Whitney U
instead, rather than just asserting it.

In [10]:
print("Normality check (D'Agostino-Pearson, alpha=0.05):\n")
for col in continuous_cols:
    result = check_normality(df[col])
    verdict = "approx. normal" if result["is_normal"] else "NOT normal"
    print(f"  {col}: p={result['p_value']:.2e} -> {verdict}")

Normality check (D'Agostino-Pearson, alpha=0.05):

  BMI: p=0.00e+00 -> NOT normal
  MentHlth: p=0.00e+00 -> NOT normal
  PhysHlth: p=0.00e+00 -> NOT normal


In [11]:
print("Variance homogeneity check (Levene's test, alpha=0.05):\n")
for col in continuous_cols:
    group0 = df.loc[df[target] == 0, col]
    group1 = df.loc[df[target] == 1, col]
    result = check_variance_homogeneity(group0, group1)
    verdict = "equal variance" if result["equal_variance"] else "UNEQUAL variance"
    print(f"  {col}: p={result['p_value']:.2e} -> {verdict}")

Variance homogeneity check (Levene's test, alpha=0.05):

  BMI: p=0.00e+00 -> UNEQUAL variance
  MentHlth: p=1.14e-267 -> UNEQUAL variance
  PhysHlth: p=0.00e+00 -> UNEQUAL variance


**Expected outcome:** all three continuous features will likely fail the
normality check (BMI is right-skewed; MentHlth/PhysHlth are zero-inflated —
most respondents report 0 days). This confirms Mann-Whitney U (rank-based,
no normality assumption) is the right choice over a t-test, as already used
in `01_eda.ipynb`.

## 3. Hypothesis Tests — Categorical Features

Running the chi-square test formally for every binary/ordinal feature against
the target, paired with Cramer's V as the effect size — a significant p-value
alone doesn't say whether the association is practically meaningful, only
that it's unlikely to be pure noise.

In [12]:
categorical_results = []
for col in categorical_cols:
    test_result = chi_square_test(df[col], df[target])
    v = cramers_v(df[col], df[target])
    categorical_results.append({
        "feature": col,
        "test": "chi-square",
        "statistic": test_result["chi2"],
        "p_value": test_result["p_value"],
        "effect_size": v,
        "effect_size_measure": "cramers_v",
        "effect_size_label": effect_size_label(v, "cramers_v"),
    })

categorical_results_df = pd.DataFrame(categorical_results).sort_values("effect_size", ascending=False).reset_index(drop=True)
categorical_results_df

,feature,test,statistic,p_value,effect_size,effect_size_measure,effect_size_label
0,GenHlth,chi-square,22728.069055,0.000000e+00,0.299296,cramers_v,medium
1,HighBP,chi-square,17562.446090,0.000000e+00,0.263110,cramers_v,medium
2,DiffWalk,chi-square,12092.319741,0.000000e+00,0.218321,cramers_v,medium
3,HighChol,chi-square,10174.074889,0.000000e+00,0.200255,cramers_v,medium
4,Age,chi-square,8795.050614,0.000000e+00,0.186072,cramers_v,small
5,HeartDiseaseorAttack,chi-square,7971.155841,0.000000e+00,0.177252,cramers_v,small
6,Income,chi-square,7003.715091,0.000000e+00,0.166075,cramers_v,small
7,Education,chi-square,4027.112282,0.000000e+00,0.125917,cramers_v,small
8,PhysActivity,chi-square,3539.419370,0.000000e+00,0.118103,cramers_v,small
9,Stroke,chi-square,2838.916547,0.000000e+00,0.105769,cramers_v,small


## 4. Hypothesis Tests — Continuous Features

Same idea for BMI, MentHlth, and PhysHlth — Mann-Whitney U as the test (per
the assumption checks above), rank-biserial correlation as the primary effect
size, with Cohen's d reported alongside only for readers used to that metric.

In [13]:
continuous_results = []
for col in continuous_cols:
    group0 = df.loc[df[target] == 0, col]
    group1 = df.loc[df[target] == 1, col]

    test_result = mann_whitney_test(group0, group1)
    r_rb = rank_biserial_from_mannwhitney(test_result["u_stat"], len(group0), len(group1))
    d = cohens_d(group0, group1)

    continuous_results.append({
        "feature": col,
        "test": "mann-whitney-u",
        "statistic": test_result["u_stat"],
        "p_value": test_result["p_value"],
        "effect_size": r_rb,
        "effect_size_measure": "rank_biserial",
        "effect_size_label": effect_size_label(r_rb, "rank_biserial"),
        "cohens_d_reference_only": d,
    })

continuous_results_df = pd.DataFrame(continuous_results).sort_values("effect_size", key=abs, ascending=False).reset_index(drop=True)
continuous_results_df

,feature,test,statistic,p_value,effect_size,effect_size_measure,effect_size_label,cohens_d_reference_only
0,BMI,mann-whitney-u,2.405335e+09,0.000000e+00,0.376633,rank_biserial,medium,-0.641442
1,PhysHlth,mann-whitney-u,2.986457e+09,0.000000e+00,0.226029,rank_biserial,small,-0.502197
2,MentHlth,mann-whitney-u,3.648124e+09,1.750549e-90,0.054551,rank_biserial,negligible,-0.200644


## 5. Multiple-Testing Correction

Running 21 tests at once means even independent, truly-null features have a
combined chance well above 5% of showing at least one 'significant' p-value
by chance alone. Benjamini-Hochberg FDR correction adjusts for that, so the
final significance calls in section 6 are trustworthy as a batch, not just
individually.

In [14]:
all_results_df = pd.concat([
    categorical_results_df[["feature", "test", "p_value", "effect_size", "effect_size_measure", "effect_size_label"]],
    continuous_results_df[["feature", "test", "p_value", "effect_size", "effect_size_measure", "effect_size_label"]],
], ignore_index=True)

correction_df = apply_multiple_testing_correction(all_results_df["p_value"].tolist(), method="fdr_bh")
all_results_df["p_value_corrected"] = correction_df["p_value_corrected"]
all_results_df["significant_after_correction"] = correction_df["significant_after_correction"]

n_raw_significant = (all_results_df["p_value"] < 0.05).sum()
n_corrected_significant = all_results_df["significant_after_correction"].sum()
print(f"Significant at raw p<0.05: {n_raw_significant} / {len(all_results_df)}")
print(f"Significant after FDR correction: {n_corrected_significant} / {len(all_results_df)}")

Significant at raw p<0.05: 21 / 21
Significant after FDR correction: 21 / 21


## 6. Consolidated Results Table

One ranked table — sorted by effect size, not p-value, since with a dataset
this large (n=253,680) even tiny, practically meaningless differences reach
statistical significance. Effect size is what actually distinguishes a
meaningful finding from statistical noise at this sample size.

In [15]:
final_table = all_results_df.sort_values("effect_size", key=abs, ascending=False).reset_index(drop=True)
final_table

,feature,test,p_value,effect_size,effect_size_measure,effect_size_label,p_value_corrected,significant_after_correction
0,BMI,mann-whitney-u,0.000000e+00,0.376633,rank_biserial,medium,0.000000e+00,True
1,GenHlth,chi-square,0.000000e+00,0.299296,cramers_v,medium,0.000000e+00,True
2,HighBP,chi-square,0.000000e+00,0.263110,cramers_v,medium,0.000000e+00,True
3,PhysHlth,mann-whitney-u,0.000000e+00,0.226029,rank_biserial,small,0.000000e+00,True
4,DiffWalk,chi-square,0.000000e+00,0.218321,cramers_v,medium,0.000000e+00,True
5,HighChol,chi-square,0.000000e+00,0.200255,cramers_v,medium,0.000000e+00,True
6,Age,chi-square,0.000000e+00,0.186072,cramers_v,small,0.000000e+00,True
7,HeartDiseaseorAttack,chi-square,0.000000e+00,0.177252,cramers_v,small,0.000000e+00,True
8,Income,chi-square,0.000000e+00,0.166075,cramers_v,small,0.000000e+00,True
9,Education,chi-square,0.000000e+00,0.125917,cramers_v,small,0.000000e+00,True


In [16]:
final_table.to_csv("../reports/statistical_test_results.csv", index=False)
print("Saved to reports/statistical_test_results.csv")

Saved to reports/statistical_test_results.csv


## 7. Interpretation

_Fill in after reviewing the table above:_
- _Which features are both statistically significant (after correction) AND
  have at least a small/medium effect size — these are the genuinely useful
  findings, not just 'technically significant' ones_
- _Any feature that was significant before correction but not after — worth
  noting as a caution against over-interpreting the raw EDA associations from
  notebook 01_
- _Any surprising result (a feature expected to matter that shows a
  negligible effect size, or vice versa)_

Next: `03_feature_engineering.ipynb` — carry forward the top-ranked features
from this notebook's table into feature engineering decisions.